# 🇭🇷 VEPRAD Croatian Speech — CTC Segmentation Pipeline

This notebook aligns VEPRAD audio recordings with their transcripts and splits them into short segments ready for ASR training using NVIDIA NeMo.

---

### 📁 Expected folder structure
```
Croatian-Language-Ctc_Segmentation/
├── VEPRAD/
│   ├── audio_m/      ← male audio
│   ├── audio_z/      ← female audio
│   └── transkripcije_m+z/   ← transcripts
└── veprad_ctc_segmentation.ipynb  ← this notebook
```

In [1]:
# Install required packages
# This takes a few minutes to complete
import sys
import subprocess

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel', 'Cython'])
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'ctc-segmentation',
    'nemo_toolkit',
    'huggingface-hub==0.23.5',
    'librosa',
    'imageio-ffmpeg',
    'transformers',
    'sentencepiece',
    'inflect',
    'webdataset',
    'pyannote.core',
    'pyannote.metrics',
    'editdistance',
    'jiwer'
])


0

In [1]:
import sys, subprocess, os, shutil, urllib.request

BRANCH = globals().get('BRANCH', 'r2.0.0rc0')
requirements_url = f'https://raw.githubusercontent.com/NVIDIA/NeMo/{BRANCH}/tools/ctc_segmentation/requirements.txt'
req_path = 'nemo_ctc_requirements.txt'
filtered_path = 'nemo_ctc_requirements.filtered.txt'
print('Downloading requirements from', requirements_url)
urllib.request.urlretrieve(requirements_url, req_path)

# Filter out ctc_segmentation (repo pins an older version that may fail to build on Windows)
with open(req_path, 'r', encoding='utf-8') as r_in:
    lines = [l for l in r_in.readlines() if 'ctc_segmentation' not in l]
with open(filtered_path, 'w', encoding='utf-8') as r_out:
    r_out.writelines(lines)

print('Installing Python requirements from', filtered_path)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', filtered_path])

# Ensure ffmpeg is available — prefer system package, fall back to Python package
if shutil.which('ffmpeg'):
    print('ffmpeg found at', shutil.which('ffmpeg'))
else:
    print('ffmpeg not found — attempting install')
    if os.name == 'nt':
        if shutil.which('choco'):
            subprocess.check_call(['choco', 'install', 'ffmpeg', '-y'])
        else:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'imageio-ffmpeg'])
            print('Installed imageio-ffmpeg (Python wrapper). If you need the ffmpeg binary, download from https://ffmpeg.org/download.html')
    else:
        try:
            subprocess.check_call(['sudo', 'apt-get', 'update'])
            subprocess.check_call(['sudo', 'apt-get', 'install', '-y', 'ffmpeg'])
        except Exception:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'imageio-ffmpeg'])
            print('Installed imageio-ffmpeg as fallback')


Installing Python requirements from nemo_ctc_requirements.filtered.txt
ffmpeg not found — attempting install
Installed imageio-ffmpeg (Python wrapper). If you need the ffmpeg binary, download from https://ffmpeg.org/download.html


In [2]:
import os
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
VEPRAD_ROOT      = "./VEPRAD"
AUDIO_DIRS       = [
    os.path.join(VEPRAD_ROOT, "audio_m"),
    os.path.join(VEPRAD_ROOT, "audio_z"),
]
TRANSCRIPT_DIR   = os.path.join(VEPRAD_ROOT, "transkripcije_m+z")
CLEAN_DIR        = "./veprad_transcripts_clean"
OUTPUT_DIR       = "./veprad_segmented"
MANIFEST_PATH    = "./veprad_manifest.json"

# ── Model ──────────────────────────────────────────────────────────────────
MODEL_NAME       = "nvidia/stt_hr_conformer_ctc_large"
SAMPLE_RATE      = 16000

# ── Segmentation settings ──────────────────────────────────────────────────
THRESHOLD        = -10      # ← changed from -10
WINDOW_LEN       = 8000

# ── Verify paths exist ─────────────────────────────────────────────────────
for d in [VEPRAD_ROOT, TRANSCRIPT_DIR] + AUDIO_DIRS:
    status = "✅" if Path(d).exists() else "❌ NOT FOUND"
    print(f"{status}  {d}")

✅  ./VEPRAD
✅  ./VEPRAD\transkripcije_m+z
✅  ./VEPRAD\audio_m
✅  ./VEPRAD\audio_z


In [3]:
transcript_files = sorted(Path(TRANSCRIPT_DIR).rglob("*.txt"))

print(f"📂 Total transcript files found: {len(transcript_files)}")
print("\n" + "─"*100)
print("📄 Previewing first 5 transcript files (raw):")
print("─"*100)

for f in transcript_files[200:205]:
    content = f.read_text(encoding="utf-8", errors="replace")
    print(f"\n▶ {f.name}")
    print(content[:400])  # show first 300 chars
    print("─"*100)

📂 Total transcript files found: 5257

────────────────────────────────────────────────────────────────────────────────────────────────────
📄 Previewing first 5 transcript files (raw):
────────────────────────────────────────────────────────────────────────────────────────────────────

▶ m01210105302.txt
osobe osjetljive na vrijeme mogu patiti od glavobolje i razdra`ljivosti
────────────────────────────────────────────────────────────────────────────────────────────────────

▶ m01210105303.txt
u ostalim ^e krajevima nepovoljan vremenski utjecaj biti slabije izra`en
────────────────────────────────────────────────────────────────────────────────────────────────────

▶ m01210203101.txt
vremenska prognoza za poslijepodne na jadranu uglavnom sun~ano
────────────────────────────────────────────────────────────────────────────────────────────────────

▶ m01210203102.txt
vjetar slab a na jadranu umjerena potkraj dana mjestimice i jaka bura <uzdah>
──────────────────────────────────────────────

---
## 🔧 Cell 4 — Encoding Fix Map
Based on what you saw above, verify or adjust the symbol mapping below.

| Symbol in file | Croatian character |
|---|---|
| `{` | `š` |
| `^` | `ć` |
| `~` | `č` |
| ` | `ž` |
| `\\` | `đ` |

**If any of these are wrong, edit the `ENCODING_MAP` dict before continuing.**

In [4]:
import re

ENCODING_MAP = {
    '{' : 'š',
    '^' : 'ć',
    '~' : 'č',
    '`' : 'ž',
}

def fix_encoding(text: str) -> str:
    for old, new in ENCODING_MAP.items():
        text = text.replace(old, new)
    return text

def clean_transcript(text: str) -> str:
    text = fix_encoding(text)
    
    # Split on silence markers
    parts = re.split(r'<sil>', text, flags=re.IGNORECASE)
    
    cleaned_lines = []
    for part in parts:
        part = re.sub(r"<[^>]+>", "", part)
        part = re.sub(r"\[[^\]]+\]", "", part)
        part = part.lower()
        part = re.sub(r"[^a-zčćđšž\s]", "", part)
        part = re.sub(r"\s+", " ", part).strip()
        if not part:
            continue
        # One word per line, skip very short words
        words = [w for w in part.split() if len(w) >= 3]
        cleaned_lines.extend(words)
    
    return "\n".join(cleaned_lines)

### Cell 5 — Save cleaned transcripts

In [5]:
import os
from pathlib import Path

os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

transcript_files = sorted(Path(TRANSCRIPT_DIR).rglob("*.txt"))

for f in transcript_files:
    raw = f.read_text(encoding="utf-8", errors="replace")
    cleaned = clean_transcript(raw)
    out_path = Path(CLEAN_DIR) / f.name
    out_path.write_text(cleaned, encoding="utf-8")

print(f"✅ Cleaned {len(transcript_files)} transcripts → {CLEAN_DIR}")

# Preview one file to verify
sample = Path(CLEAN_DIR) / transcript_files[0].name
print(f"\n📄 Sample: {sample.name}")
print(sample.read_text(encoding="utf-8"))

✅ Cleaned 5257 transcripts → ./veprad_transcripts_clean

📄 Sample: m01020105401.txt
biometeorološka
prognoza
sutra
uvjeti
biti
razmjerno
povoljni


In [6]:
import nemo.collections.asr as nemo_asr

asr_model = nemo_asr.models.EncDecCTCModel.from_pretrained(model_name=MODEL_NAME)
asr_model.eval()

char_list = asr_model.decoder.vocabulary
print(f"✅ Model loaded. Vocab size: {len(char_list)}")
print("First 20 tokens:", char_list[:20])

[NeMo W 2026-05-19 23:43:01 transformer_bpe_models:59] Could not import NeMo NLP collection which is required for speech translation model.


[NeMo I 2026-05-19 23:43:06 mixins:170] Tokenizer SentencePieceTokenizer initialized with 128 tokens


[NeMo W 2026-05-19 23:43:06 modelPT:161] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    sample_rate: 16000
    batch_size: 16
    shuffle: true
    num_workers: 8
    pin_memory: true
    use_start_end_token: ''
    trim_silence: false
    max_duration: 20.0
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: synced_randomized
    bucketing_batch_size: null
    
[NeMo W 2026-05-19 23:43:06 modelPT:168] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath: null
    sample_rate: 16000
    batch_size: 16
    shuffle: false
    num_workers: 8
    pin_

[NeMo I 2026-05-19 23:43:06 features:289] PADDING: 0
[NeMo I 2026-05-19 23:43:10 save_restore_connector:249] Model EncDecCTCModelBPE was successfully restored from C:\Users\Lukas\.cache\huggingface\hub\models--nvidia--stt_hr_conformer_ctc_large\snapshots\d49f834e3afe270c11b16523be98b9f3f29fb406\stt_hr_conformer_ctc_large.nemo.
✅ Model loaded. Vocab size: 128
First 20 tokens: ['<unk>', '▁', 'i', 'u', 's', 'je', 'm', 'a', 'o', 'p', 'd', 'ni', '▁i', 'ra', 't', 'n', 'no', 'e', 'v', '▁o']


### CTC Segmentation

In [7]:
import torch
import soundfile as sf
import numpy as np
from ctc_segmentation import (
    CtcSegmentationParameters,
    ctc_segmentation,
    prepare_text,
    determine_utterance_segments,
)

def get_log_probs(model, audio_path, sample_rate=16000):
    audio, sr = sf.read(audio_path, dtype="float32")
    assert sr == sample_rate, f"Expected {sample_rate} Hz, got {sr} Hz"
    
    device = next(model.parameters()).device
    audio_tensor = torch.tensor(audio).unsqueeze(0).to(device)
    audio_len    = torch.tensor([audio_tensor.shape[1]]).to(device)

    with torch.no_grad():
        log_probs_batch, lengths, greedy_predictions = model(
            input_signal=audio_tensor,
            input_signal_length=audio_len
        )
    return log_probs_batch[0].cpu().numpy()


def segment_file(model, audio_path, transcript_path, char_list):
    utterances = [
        l.strip() for l in
        Path(transcript_path).read_text(encoding="utf-8").splitlines()
        if l.strip()
    ]
    if not utterances:
        return []

    lpz = get_log_probs(model, audio_path)

    config = CtcSegmentationParameters()
    config.char_list       = list(char_list)  # plain Python list
    config.min_window_size = WINDOW_LEN
    config.index_duration  = 0.04
    config.blank           = 0

    ground_truth_mat, utt_begin_indices = prepare_text(config, utterances)
    timings, char_probs, state_list = ctc_segmentation(config, lpz, ground_truth_mat)

    segments = determine_utterance_segments(
        config, utt_begin_indices, char_probs, timings, utterances  # ← back in
    )

    result = []
    for utt, seg in zip(utterances, segments):
        start, end, score = seg[0], seg[1], seg[2]
        result.append((utt, (start, end, score)))
    return result

In [8]:
raw_txt = sorted(Path(TRANSCRIPT_DIR).rglob("*.txt"))[0]
print(raw_txt.name)
print(raw_txt.read_text(encoding="utf-8", errors="replace"))

m01020105401.txt
biometeorolo{ka prognoza za sutra <sil> uvjeti ^e biti razmjerno povoljni


In [9]:
import json
import soundfile as sf
import numpy as np
from tqdm import tqdm

manifest_entries = []

for audio_dir in AUDIO_DIRS:
    audio_files = sorted(Path(audio_dir).rglob("*.wav"))[:10]
    print(f"\n🔊 Processing {len(audio_files)} files in {audio_dir}")

    for audio_path in tqdm(audio_files):
        stem = audio_path.stem
        clean_txt = Path(CLEAN_DIR) / f"{stem}.txt"

        if not clean_txt.exists():
            print(f"  ⚠️  No transcript for {stem}, skipping.")
            continue

        try:
            results = segment_file(asr_model, audio_path, clean_txt, char_list)
        except Exception as e:
            print(f"  ❌ Failed {stem}: {e}")
            continue

        audio, sr = sf.read(str(audio_path), dtype="float32")
        out_stem_dir = Path(OUTPUT_DIR) / stem
        out_stem_dir.mkdir(parents=True, exist_ok=True)

        for idx, (text, (start, end, score)) in enumerate(results):
            if score < THRESHOLD:
                continue
            if end - start < 0.1:
                continue

            s_frame = int(start * sr)
            e_frame = int(end   * sr)
            clip    = audio[s_frame:e_frame]

            out_wav = out_stem_dir / f"{stem}_{idx:04d}.wav"
            sf.write(str(out_wav), clip, sr)

            manifest_entries.append({
                "audio_filepath": str(out_wav.resolve()),
                "text":           text,
                "duration":       round(end - start, 4),
                "offset_start":   round(float(start), 4),   # ← added
                "offset_end":     round(float(end), 4),     # ← added
                "score":          round(float(score), 4),
                "source_file":    str(audio_path.resolve()), # ← optional but useful
            })

with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    for entry in manifest_entries:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"\n✅ Done! {len(manifest_entries)} segments → {MANIFEST_PATH}")


🔊 Processing 10 files in ./VEPRAD\audio_m


  0%|          | 0/10 [00:00<?, ?it/s][NeMo W 2026-05-19 23:43:43 nemo_logging:349] c:\Users\Lukas\.julia\conda\3\lib\site-packages\nemo\collections\asr\parts\preprocessing\features.py:417: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
      with torch.cuda.amp.autocast(enabled=False):
    
100%|██████████| 10/10 [00:10<00:00,  1.10s/it]



🔊 Processing 10 files in ./VEPRAD\audio_z


100%|██████████| 10/10 [00:02<00:00,  4.92it/s]


✅ Done! 47 segments → ./veprad_manifest.json
